In [1]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
plt.style.use('ggplot')
from matplotlib.pyplot import figure
%matplotlib inline
matplotlib.rcParams['figure.figsize'] = (12,8)
import requests
import imdb
import time
from tqdm import tqdm
from imdb import IMDb
import random
import unicodedata
import re

In [2]:
df = pd.read_csv(r'C:\Users\Aishw\Documents\movies.csv')

In [3]:
df_original = df.copy()

In [4]:
df.head()

,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,runtime
0,The Shining,R,Drama,1980,"June 13, 1980 (United States)",8.4,927000.0,Stanley Kubrick,Stephen King,Jack Nicholson,United Kingdom,19000000.0,46998772.0,Warner Bros.,146.0
1,The Blue Lagoon,R,Adventure,1980,"July 2, 1980 (United States)",5.8,65000.0,Randal Kleiser,Henry De Vere Stacpoole,Brooke Shields,United States,4500000.0,58853106.0,Columbia Pictures,104.0
2,Star Wars: Episode V - The Empire Strikes Back,PG,Action,1980,"June 20, 1980 (United States)",8.7,1200000.0,Irvin Kershner,Leigh Brackett,Mark Hamill,United States,18000000.0,538375067.0,Lucasfilm,124.0
3,Airplane!,PG,Comedy,1980,"July 2, 1980 (United States)",7.7,221000.0,Jim Abrahams,Jim Abrahams,Robert Hays,United States,3500000.0,83453539.0,Paramount Pictures,88.0
4,Caddyshack,R,Comedy,1980,"July 25, 1980 (United States)",7.3,108000.0,Harold Ramis,Brian Doyle-Murray,Chevy Chase,United States,6000000.0,39846344.0,Orion Pictures,98.0


In [5]:
df.dtypes

name         object
rating       object
genre        object
year          int64
released     object
score       float64
votes       float64
director     object
writer       object
star         object
country      object
budget      float64
gross       float64
company      object
runtime     float64
dtype: object

,name,rating,genre,released,director,writer,star,country,company
count,7668,7591,7668,7666,7668,7665,7667,7665,7651
unique,7512,12,19,3414,2949,4535,2814,59,2385
top,Nobody's Fool,R,Comedy,"October 4, 1991 (United States)",Woody Allen,Woody Allen,Nicolas Cage,United States,Universal Pictures
freq,3,3697,2245,9,38,37,43,5475,377


In [6]:
print(df.isnull().sum())

name           0
rating        77
genre          0
year           0
released       2
score          3
votes          3
director       0
writer         3
star           1
country        3
budget      2171
gross        189
company       17
runtime        4
dtype: int64


In [8]:
#Start with data cleaning
#Changing the released column data type to date time format
df['released'].dtypes
print("Count of missing values : {}".format(df['released'].isna().sum()))

Count of missing values : 2


In [9]:
#changing data type of release column
df['released_cleaned'] = df['released'].str.replace(r"\s*\(.*\)", "", regex=True)

In [10]:
df['released_date'] = pd.to_datetime(df['released_cleaned'], errors='coerce')

In [11]:
df.drop(columns=['released_cleaned','released'], inplace=True)

In [12]:
print(df['released_date'].head())
df['released_date'].dtypes

0   1980-06-13
1   1980-07-02
2   1980-06-20
3   1980-07-02
4   1980-07-25
Name: released_date, dtype: datetime64[ns]


dtype('<M8[ns]')

In [29]:
#Changing rating column data type to categorical variable 
#Replacing the value 'Unrated' as 'Not Rated' as both means the same
df['rating'].value_counts()

rating
R            3709
PG-13        2112
PG           1253
Not Rated     345
G             153
Unrated        52
NC-17          23
TV-MA          10
TV-PG           5
X               3
Approved        1
18+             1
TV-14           1
Name: count, dtype: int64

In [13]:
df['rating'] = df['rating'].replace({
    'Unrated': 'Not Rated'})

In [14]:
df['rating'] = df['rating'].astype('category')

In [15]:
df['rating'].value_counts()

rating
R            3697
PG-13        2112
PG           1252
Not Rated     335
G             153
NC-17          23
TV-MA           9
TV-PG           5
X               3
Approved        1
TV-14           1
Name: count, dtype: int64

In [16]:
df['released_year'] = pd.to_datetime(df['released_date'], errors='coerce').dt.year

In [17]:
#Clean the movie title column and change the column name to title
def cleanmoviename(name):
    if pd.isnull(name):
        return name
    # Ensure it's a string
    name = str(name)

    # Normalize unicode characters
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('utf-8')
    
    # Remove unwanted special characters (keep letters, numbers, spaces, dashes, colons, apostrophes)
    name = re.sub(r"[^a-zA-Z0-9\s\-\:\']+", '', name)

    # Replace multiple spaces with single space
    name = re.sub(r"\s+", ' ', name)

    # Strip leading/trailing spaces
    name = name.strip()

    return name

In [18]:
# Apply cleaning function
df['name'] = df['name'].apply(cleanmoviename)

In [19]:
# Rename the column to 'Title'
df.rename(columns={'name': 'Title'}, inplace=True)

In [21]:
df.dtypes

Title                    object
rating                 category
genre                    object
year                      int64
score                   float64
votes                   float64
director                 object
writer                   object
star                     object
country                  object
budget                  float64
gross                   float64
company                  object
runtime                 float64
released_date    datetime64[ns]
released_year           float64
dtype: object

In [22]:
df.describe(include=[np.number])

,year,score,votes,budget,gross,runtime,released_year
count,7668.000000,7665.000000,7.665000e+03,5.497000e+03,7.479000e+03,7664.000000,7609.000000
mean,2000.405451,6.390411,8.810850e+04,3.558988e+07,7.850054e+07,107.261613,2000.700486
std,11.153508,0.968842,1.633238e+05,4.145730e+07,1.657251e+08,18.581247,11.152091
min,1980.000000,1.900000,7.000000e+00,3.000000e+03,3.090000e+02,55.000000,1980.000000
25%,1991.000000,5.800000,9.100000e+03,1.000000e+07,4.532056e+06,95.000000,1991.000000
50%,2000.000000,6.500000,3.300000e+04,2.050000e+07,2.020576e+07,104.000000,2001.000000
75%,2010.000000,7.100000,9.300000e+04,4.500000e+07,7.601669e+07,116.000000,2010.000000
max,2020.000000,9.300000,2.400000e+06,3.560000e+08,2.847246e+09,366.000000,2020.000000


In [ ]:
#sample code to get the movie data using OMDb API

api_key = "48262c0f"
title = "The Matrix"
url = f"http://www.omdbapi.com/?t={title}&apikey={api_key}"

response = requests.get(url)
data = response.json()
print(data)
print("Title:", data.get('Title'))
print("IMDB Rating:", data.get('imdbRating'))
print("Votes:", data.get('imdbVotes'))
print("Genre:", data.get('Genre'))
print("Country:", data.get('Country'))

In [ ]:
# API key
OMDB_API_KEY = "48262c0f"

# Target fields to fill (excluding 'gross' for now)
columns_to_fill = ['score', 'votes', 'writer', 'star', 'country', 'rating']

def fetch_movie_data_omdb(title):
    url = f"http://www.omdbapi.com/?t={title}&apikey={OMDB_API_KEY}"
    try:
        response = requests.get(url, timeout=5)
        data = response.json()
        if data.get("Response") == "True":
            return data
        else:
            return None
    except Exception as e:
        print(f"Error fetching '{title}': {e}")
        return None

def fill_missing_from_omdb(df):
    failed_titles = []

    # Only rows where at least one column is missing (not including gross)
    rows_to_update = df[df[columns_to_fill].isna().any(axis=1)]

    for idx, row in tqdm(rows_to_update.iterrows(), total=rows_to_update.shape[0]):
        title = row['Title']
        data = fetch_movie_data_omdb(row['Title'])
        if not data:
            failed_titles.append(title)
            continue

        if pd.isna(df.at[idx, 'score']):
            df.at[idx, 'score'] = float(data.get('imdbRating')) if data.get('imdbRating') != 'N/A' else None

        if pd.isna(df.at[idx, 'votes']):
            votes = data.get('imdbVotes')
            if votes and votes != 'N/A':
                df.at[idx, 'votes'] = int(votes.replace(',', ''))

        if pd.isna(df.at[idx, 'writer']):
            writer = data.get('Writer')
            if writer and writer != 'N/A':
                df.at[idx, 'writer'] = writer.split(',')[0]  # Take first writer

        if pd.isna(df.at[idx, 'star']):
            actor = data.get('Actors')
            if actor and actor != 'N/A':
                df.at[idx, 'star'] = actor.split(',')[0]  # Take first star

        if pd.isna(df.at[idx, 'country']):
            country = data.get('Country')
            if country and country != 'N/A':
                df.at[idx, 'country'] = country.split(',')[0]

        # Fill MPAA rating
        if pd.isna(df.at[idx, 'rating']):
            rated = data.get('Rated')
            if rated and rated != 'N/A':
                df.at[idx, 'rating'] = rated

        time.sleep(0.2)  # Slight delay to avoid hitting request limits

    print(f"\n🔁 Retry needed for {len(failed_titles)} titles.")
    return df, failed_titles


In [36]:
df, failed_titles = fill_missing_from_omdb(df)

100%|██████████| 4/4 [00:03<00:00,  1.21it/s]


🔁 Retry needed for 1 titles.


In [37]:
print(failed_titles)

['Saw: The Final Chapter']


In [38]:
df['rating'].fillna("Not Rated", inplace=True)

C:\Users\Aishw\AppData\Local\Temp\ipykernel_17880\1134614394.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['rating'].fillna("Not Rated", inplace=True)


In [39]:
def fill_gross_from_omdb(df, api_key="48262c0f"):
    missing_gross = df[df['gross'].isna()]
    failed_titles = []
    
    for idx, row in tqdm(missing_gross.iterrows(), total=missing_gross.shape[0]):
        title = row['Title']
        url = f"http://www.omdbapi.com/?t={title}&apikey={api_key}"
        
        try:
            response = requests.get(url, timeout=5)
            data = response.json()

            if data.get("Response") == "True":
                box_office = data.get('BoxOffice')
                if box_office and box_office != 'N/A':
                    gross_value = int(box_office.replace('$', '').replace(',', ''))
                    df.at[idx, 'gross'] = gross_value
                else:
                    failed_titles.append(title)
            else:
                failed_titles.append(title)

        except Exception as e:
            print(f"Error fetching gross for '{title}': {e}")
            failed_titles.append(title)

        time.sleep(0.2)  # Respect API limits

    print(f"\n⚠️  Could not fill 'gross' for {len(failed_titles)} titles.")
    return df, failed_titles

In [40]:
df, failed_titles = fill_gross_from_omdb(df)

100%|██████████| 90/90 [00:35<00:00,  2.55it/s]


⚠️  Could not fill 'gross' for 90 titles.


In [19]:
print(failed_titles)

['Raise the Titanic', 'Breaker Morant', 'The Boogey Man', 'Lion of the Desert', "Can't Stop the Music", 'Hangar 18', "It's My Turn", 'Moscow Does Not Believe in Tears', 'Windwalker', 'Dead Buried', 'Looker', 'Graduation Day', 'American Pop', 'An Eye for an Eye', 'Longshot', 'Time Walker', 'Love Child', 'Boardinghouse', 'Sleepaway Camp', 'Better Late Than Never', 'White Star', 'Deadly Force', 'Cross Country', 'A Polish Vampire in Burbank', 'The Business of Show Business', 'Ghoulies', 'Cloak Dagger', 'Kaos', 'FleshBlood', 'Creature', 'Bal na vodi', 'Bliss', 'On the Edge', 'The Frog Prince', 'Lune de miel', 'Mr Love', 'Death of an Angel', 'Man Facing Southeast', 'Armed Response', 'Cherry 2000', 'Silent Night Deadly Night Part 2', 'Creepozoids', 'Walker', 'Killer Klowns from Outer Space', "Jack's Back", 'Not of This Earth', 'A Summer Story', 'Wicked Stepmother', 'The Unbelievable Truth', 'Damned River', 'Side Out', 'Boiling Point', 'Heaven and Earth', 'Archangel', 'The Pit and the Pendulum

In [41]:
print(df.isnull().sum())

Title               0
rating              0
genre               0
year                0
released            2
score               1
votes               0
director            0
writer              1
star                0
country             2
budget           2171
gross              90
company            17
runtime             4
released_date      59
released_year      59
dtype: int64


In [42]:
df.to_csv('dataset_prep.csv', index=False)